# PPO RL Execution Agent
Trains a Proximal Policy Optimization (PPO) agent to optimize order execution
in a simulated LOB (limit order book) environment. The agent learns to minimize
market impact and slippage when executing large orders.

In [ ]:
!pip install -q torch numpy pandas gymnasium matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from collections import deque
import warnings
warnings.filterwarnings('ignore')

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
TOTAL_STEPS = 500_000
UPDATE_FREQ = 2048
N_EPOCHS    = 10
BATCH_SIZE  = 64
GAMMA       = 0.99
GAE_LAMBDA  = 0.95
CLIP_EPS    = 0.2
LR          = 3e-4
ENT_COEF    = 0.01
VF_COEF     = 0.5

print(f'Device: {DEVICE} | Total steps: {TOTAL_STEPS:,}')

In [ ]:
class OrderBookEnv(gym.Env):
    """
    Simulated orderbook execution environment.
    State:  [remaining_shares_pct, time_remaining_pct, mid_price_norm,
             spread_bps, bid_depth_pct, ask_depth_pct, volatility_1min,
             last_fill_slippage_bps]
    Actions: 0=PASSIVE (post limit), 1=ACTIVE (cross spread), 2=CANCEL_WAIT
    Reward:  negative slippage bps + time-pressure penalty
    """

    OBS_DIM = 8
    N_ACTIONS = 3

    def __init__(self, order_size: float = 1000.0, time_horizon: int = 60):
        super().__init__()
        self.order_size   = order_size
        self.time_horizon = time_horizon
        self.observation_space = spaces.Box(-np.inf, np.inf, (self.OBS_DIM,), np.float32)
        self.action_space      = spaces.Discrete(self.N_ACTIONS)
        self.reset()

    def _generate_lob_snapshot(self):
        """Simulates a realistic LOB snapshot with correlated spread/depth."""
        self._mid_price = self._mid_price * (1 + np.random.normal(0, 0.0002))
        spread_bps      = max(0.5, np.random.exponential(3.0))
        bid_depth       = np.random.uniform(0.1, 1.0)
        ask_depth       = np.random.uniform(0.1, 1.0)
        return spread_bps, bid_depth, ask_depth

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._mid_price    = 100.0 + np.random.normal(0, 10)
        self._remaining    = self.order_size
        self._step         = 0
        self._last_slippage = 0.0
        self._vol_1min     = abs(np.random.normal(0.001, 0.0005))
        return self._get_obs(), {}

    def _get_obs(self) -> np.ndarray:
        spread_bps, bid_depth, ask_depth = self._generate_lob_snapshot()
        return np.array([
            self._remaining / self.order_size,
            1.0 - self._step / self.time_horizon,
            self._mid_price / 100.0 - 1.0,
            spread_bps / 20.0,
            bid_depth,
            ask_depth,
            self._vol_1min * 100,
            self._last_slippage / 50.0,
        ], dtype=np.float32)

    def step(self, action: int):
        self._step += 1
        spread_bps, bid_depth, ask_depth = self._generate_lob_snapshot()

        if action == 0:   # PASSIVE — post limit, risk of non-fill
            fill_prob   = bid_depth * 0.7
            filled      = self._remaining * fill_prob * np.random.uniform(0.3, 0.7)
            slippage_bps = np.random.uniform(-1, spread_bps * 0.3)
        elif action == 1: # ACTIVE — market order, always fills
            filled      = min(self._remaining, self.order_size * 0.15)
            slippage_bps = spread_bps * 0.5 + np.random.normal(0, 0.5)
        else:             # CANCEL/WAIT
            filled      = 0.0
            slippage_bps = 0.0

        self._remaining     = max(0, self._remaining - filled)
        self._last_slippage = slippage_bps

        # Reward: penalise slippage; also penalise time pressure if behind schedule
        expected_rate   = self.order_size / self.time_horizon
        actual_rate     = (self.order_size - self._remaining) / max(self._step, 1)
        time_penalty    = max(0, expected_rate - actual_rate) * 0.01
        reward          = -slippage_bps - time_penalty

        terminated = self._remaining <= 0
        truncated  = self._step >= self.time_horizon

        if truncated and self._remaining > 0:
            # Forced market sweep penalty
            reward -= self._remaining / self.order_size * 50

        return self._get_obs(), reward, terminated, truncated, {}


env = OrderBookEnv()
obs, _ = env.reset()
print(f'Obs shape: {obs.shape} | Action space: {env.action_space.n}')

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.actor  = nn.Linear(hidden, n_actions)
        self.critic = nn.Linear(hidden, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor):
        h      = self.shared(x)
        logits = self.actor(h)
        value  = self.critic(h).squeeze(-1)
        return logits, value

    def act(self, obs: np.ndarray):
        x          = torch.tensor(obs, dtype=torch.float32).to(DEVICE)
        logits, v  = self(x)
        dist       = Categorical(logits=logits)
        action     = dist.sample()
        return action.item(), dist.log_prob(action).item(), v.item()


agent     = ActorCritic(OrderBookEnv.OBS_DIM, OrderBookEnv.N_ACTIONS).to(DEVICE)
optimizer = optim.Adam(agent.parameters(), lr=LR, eps=1e-5)
print(f'Agent params: {sum(p.numel() for p in agent.parameters()):,}')

In [ ]:
def ppo_update(rollout):
    obs_t   = torch.tensor(np.array(rollout['obs']),    dtype=torch.float32).to(DEVICE)
    acts_t  = torch.tensor(rollout['actions'],          dtype=torch.long).to(DEVICE)
    logps_t = torch.tensor(rollout['log_probs'],        dtype=torch.float32).to(DEVICE)
    rets_t  = torch.tensor(rollout['returns'],          dtype=torch.float32).to(DEVICE)
    adv_t   = torch.tensor(rollout['advantages'],       dtype=torch.float32).to(DEVICE)
    adv_t   = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-8)

    n = len(obs_t)
    idxs = np.arange(n)
    for _ in range(N_EPOCHS):
        np.random.shuffle(idxs)
        for start in range(0, n, BATCH_SIZE):
            mb = idxs[start:start + BATCH_SIZE]
            logits, values = agent(obs_t[mb])
            dist        = Categorical(logits=logits)
            new_log_probs = dist.log_prob(acts_t[mb])
            entropy     = dist.entropy().mean()
            ratio       = torch.exp(new_log_probs - logps_t[mb])
            surr1       = ratio * adv_t[mb]
            surr2       = torch.clamp(ratio, 1-CLIP_EPS, 1+CLIP_EPS) * adv_t[mb]
            policy_loss = -torch.min(surr1, surr2).mean()
            value_loss  = nn.MSELoss()(values, rets_t[mb])
            loss        = policy_loss + VF_COEF * value_loss - ENT_COEF * entropy
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), 0.5)
            optimizer.step()


def compute_gae(rewards, values, dones):
    advantages = np.zeros_like(rewards)
    last_adv   = 0
    for t in reversed(range(len(rewards))):
        next_val = values[t+1] if t < len(values)-1 else 0
        delta    = rewards[t] + GAMMA * next_val * (1 - dones[t]) - values[t]
        last_adv = delta + GAMMA * GAE_LAMBDA * (1 - dones[t]) * last_adv
        advantages[t] = last_adv
    return advantages


# Training loop
obs, _ = env.reset()
ep_rewards = deque(maxlen=100)
ep_reward  = 0.0
rollout    = {'obs': [], 'actions': [], 'log_probs': [], 'rewards': [], 'values': [], 'dones': []}
step_count = 0
update_count = 0

while step_count < TOTAL_STEPS:
    action, log_prob, value = agent.act(obs)
    next_obs, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

    rollout['obs'].append(obs.copy())
    rollout['actions'].append(action)
    rollout['log_probs'].append(log_prob)
    rollout['rewards'].append(reward)
    rollout['values'].append(value)
    rollout['dones'].append(float(done))

    ep_reward += reward
    step_count += 1

    if done:
        ep_rewards.append(ep_reward)
        ep_reward = 0.0
        obs, _ = env.reset()
    else:
        obs = next_obs

    if len(rollout['obs']) >= UPDATE_FREQ:
        advantages = compute_gae(
            np.array(rollout['rewards']),
            np.array(rollout['values']),
            np.array(rollout['dones']),
        )
        rollout['returns'] = advantages + np.array(rollout['values'])
        rollout['advantages'] = advantages
        ppo_update(rollout)
        rollout = {'obs': [], 'actions': [], 'log_probs': [], 'rewards': [], 'values': [], 'dones': []}
        update_count += 1

        if update_count % 20 == 0:
            mean_r = np.mean(ep_rewards) if ep_rewards else float('nan')
            print(f'Steps: {step_count:>7,} | Updates: {update_count:>4} | Mean reward (100 eps): {mean_r:.2f}')

print('Training complete!')

In [ ]:
# Evaluation: run 500 episodes and compute mean slippage
agent.eval()
eval_rewards = []
for _ in range(500):
    obs, _ = env.reset()
    total_r = 0
    done = False
    while not done:
        action, _, _ = agent.act(obs)
        obs, r, terminated, truncated, _ = env.step(action)
        total_r += r
        done = terminated or truncated
    eval_rewards.append(total_r)

print(f'Eval over 500 eps | Mean reward: {np.mean(eval_rewards):.2f} ± {np.std(eval_rewards):.2f}')

# Save model
save_path = 'ppo_execution_agent.pt'
torch.save(agent.state_dict(), save_path)
import os; print(f'Saved to {save_path} ({os.path.getsize(save_path)/1024:.1f} KB)')